In [1]:
%pip install requests pandas openpyxl argparse
import getpass
import argparse
import json
import os
import re
import threading
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
import pandas as pd
import requests

  Using cached argparse-1.4.0-py2.py3-none-any.whl.metadata (2.8 kB)
Using cached argparse-1.4.0-py2.py3-none-any.whl (23 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


# STEP 1: Settings

In [2]:
HOST  = "https://maki.uni-mannheim.de/v1"
MODEL = "gemma4-26b"
TEMPERATURE = 0           # 0 = deterministic / predictable
SEED = 123
MAX_WORKERS = 2              # Parallel calls; lower it if the server returns errors
TIMEOUT = 180                # Seconds to wait for one answer
MAX_RETRIES = 3              # Tries per paragraph before it is marked as an error
STOP_AFTER_ERRORS = 20       # Pause the whole run if this many fail in a row

# Prompt for API Key
API_KEY = os.environ.get("UNI_API_KEY") or getpass.getpass("Paste the Uni API key: ")
print("Key loaded:", bool(API_KEY))

Key loaded: True


# STEP 2: The Annotation Prompt

In [8]:
if "API_KEY" not in globals():
    raise RuntimeError("Please run STEP 1 first.")

# Customize your labels and definition here
ANNOTATION_PROMPT = """
ROLE 
You are a text annotator for media content analysis. Apply the Stereotype Content Model (SCM) to decide whether [Gruppe 1] is portrayed with Warmth (the opposite of Coldness) in German newspaper articles.

CONTEXT 
Input: several consecutive paragraphs from one article plus the target group [Gruppe 1]. Return one label for the whole passage. Entity names are masked as [Gruppe 1] and [andere Gruppe]; keep them exactly as written, also in your explanation.

TASK 
Label TRUE if the text portrays [Gruppe 1] or a member of [Gruppe 1] as the ACTOR of a warm trait or warm behavior toward others, shown by an explicit marker. Otherwise label FALSE. 
Others (the receiver) can be anyone: [andere Gruppe], individuals, the public, or [Gruppe 1]'s own group. 
Membership must be stated or be unmistakable in the text. If you have to guess that someone belongs to [Gruppe 1], they do not count. Pronouns (they, their) count when the antecedent in the passage is clearly [Gruppe 1]: in "They are calling on their citizens to welcome newcomers", "They" counts if the previous sentence names [Gruppe 1]. 
Others may comment on or praise [Gruppe 1] (see QUOTES), but the warm trait or behavior must be attributed to [Gruppe 1] or a member as the actor. 
Judge as a general reader: do not decide whether the author or a quoted speaker is right, only what the text conveys about [Gruppe 1].

DEFINITION
Warmth = perceived positive traits or positive behavior of [Gruppe 1] toward others, in two dimensions:
Friendliness: traits such as friendly, warm, sociable, welcoming, pleasant, kind or helpful (but not limited to these), or treating others in such a way.
Morality: traits such as moral, trustworthy, honest, sincere, fair, tolerant, altruistic or benevolent (but not limited to these), or treating others honestly, fairly, helpfully or benevolently.

EXPLICIT MARKERS
(a) A warm trait word (or synonym / noun form) applied to [Gruppe 1] or a member, e.g. "welcoming", "their honesty". 
(b) A concretely described warm act whose friendly, kind, honest, fair or altruistic nature is stated or unmistakable (see WARM BEHAVIOR).
(c) An explicit positive moral evaluation, e.g. "generously donated resources", "acted selflessly". 
(d) Emotive or hyperbolic framing of aid, rescue, charity or solidarity (see HEROISM; RESCUE).

EVIDENCE LEVEL: 
EXPLICIT ONLY 
Only explicit markers (a) to (d) count. Implicit cues (tone, positive insinuation, loaded warm wording without a trait or act, hyperbole or emotive wording on its own outside aid/rescue or Heroism/Admiration reporting, calls to action whose benevolent or prosocial aim is only implied, positive stereotypes, praise by association, hints at good motives) are NOT coded: if they are the only evidence, S1 is F.

WARM BEHAVIOR 
A behavior counts only if it is a real action with a tangible positive effect or clearly benevolent/friendly intent: helping, assisting or supporting others; donating, sharing resources or rescuing people in need; treating others fairly, honestly or hospitably; protecting others' interests, property or reputation; acting to uphold moral values or promote solidarity. It does not count if it is trivial (saying hello, polite small talk, accepting an invitation), routine compliance or self-interest (doing one's paid job normally, following standard rules, pursuing mutual business profit), cultural, religious or lifestyle difference, or an act with no evident friendly, kind, honest, fair or altruistic intent.

HEROISM, HYPERBOLE AND ADMIRATION 
Purely competence-, fame- or courage-based praise, athletic/artistic idolization, or heroic nicknames without prosocial or moral framing (celebrating skill, success, dominance, bravery, or popularity alone) is not warmth: "The [Gruppe 1] superstars dominated the tournament." or "Fans worship the [Gruppe 1] legend as an untouchable icon." However, heroic labels, hyperbole, admiration or nicknames that explicitly attribute prosocial sacrifice, protection of others, moral courage, or benevolence to [Gruppe 1] or a member ("The [Gruppe 1] saviors risked everything to shield the innocent", "Locals hailed the [Gruppe 1] doctor as a 'guardian angel' of the poor") as well as explicit emotive praise of their kindness, generosity or humanity are warmth, also inside quotes: "These [Gruppe 1] heroes are the true heart of our community, always putting others first." or "The [Gruppe 1] icon is celebrated for her boundless empathy." If a competence-only heroic label and a prosocial/warm sentence appear for the same actor or act, the warm/emotive sentence decides. Admiration for status, wealth, talent or physical bravery alone does not count. Outside heroism/admiration reporting, hyperbole or emotive wording about [Gruppe 1] counts ONLY when it is attached to a stated warm act, trait or moral evaluation (markers (a) to (c)); on its own it is NOT coded (S1 is F).

HISTORICAL, RELIGIOUS AND PHILOSOPHICAL CONTEXTS (overrides WARM BEHAVIOR)
A purely dry, encyclopedic or historical report of past events, religious doctrine, or traditional practices without prosocial framing (who, what, when, where, historical timelines, or theological exposition) is not warmth: "The ancient [Gruppe 1] established city-states across the Mediterranean." or "In early [Gruppe 1] theology, charity was considered a central virtue." However, clear portrayals of [Gruppe 1] as active agents of mercy, hospitality, charity or moral care toward others ("The ancient [Gruppe 1] sheltered travelers and protected the vulnerable", "Early [Gruppe 1] communities shared all their possessions with the poor") as well as explicit emotive labels, hyperbole, moral praise, attributed benevolent intent or celebratory framing aimed at [Gruppe 1] or a historical/religious member are warmth, also inside quotes: "These early [Gruppe 1] thinkers were beacons of tolerance and human kindness." or "The [Gruppe 1] saint is remembered for tirelessly healing the sick." If a dry historical/doctrinal statement and a warm/emotive sentence appear for the same group or act, the warm/emotive sentence decides. Praise for an abstract religious concept, text, or deity alone—without attributing the warm trait or behavior to [Gruppe 1] as human actors—does not count. Outside historical/religious reporting, hyperbole or emotive wording about [Gruppe 1] counts ONLY when it is attached to a stated warm act, trait or moral evaluation (markers (a) to (c)); on its own it is NOT coded (S1 is F).

AID, RESCUE AND SOLIDARITY 
A purely dry/bureaucratic report of routine duty or official procedure without prosocial framing (who, what, when, where, administrative or institutional steps) is not warmth: "The [Gruppe 1] official processed 40 applications on Monday afternoon." However, clear acts of aid, rescue, donation or solidarity ("The [Gruppe 1] saved the people", "The [Gruppe 1] native collected donations for the refugees") as well as explicit emotive labels, hyperbole, moral praise, attributed benevolent intent or celebratory framing aimed at [Gruppe 1] or a member are warmth, also inside quotes: "These [Gruppe 1] volunteers are tirelessly opening their doors to everyone in need." or "[Gruppe 1] helpers are turning the neighborhood into a haven of solidarity." If a dry and a warm/emotive sentence appear for the same acts, the warm/emotive sentence decides. Emotion about the recipients or the situation alone does not count. Outside aid/rescue reporting, hyperbole or emotive wording about [Gruppe 1] counts ONLY when it is attached to a stated warm act, trait or moral evaluation (markers (a) to (c)); on its own it is NOT coded (S1 is F).


QUOTES 
Warmth may come from the source or from any quoted speaker (including [andere Gruppe] or [Gruppe 1] itself), if it attributes the warm trait or behavior to [Gruppe 1] or a member as the actor. It does not count when the warm word describes someone else: "Filip takes care of everyone," says the [Gruppe 1].

NOT WARMTH 
Step 2 (actor): others are warm toward [Gruppe 1]; [Gruppe 1] is only a beneficiary or recipient ("The local population showed great readiness to help the [Gruppe 1]"); the warm trait or behavior belongs to someone not identifiable as [Gruppe 1]; [Gruppe 1] is a place or geographic location ("[Gruppe 1] is where most of the people found their purpose"). Step 3 (content): competence or courage only ("brave", "hard-working", "skilled"); benefit frame only ("[Gruppe 1] strengthen the economy"); trivial politeness or routine self-interested/paid duty; facts and statistics only ("In 2022, 12,000 [Gruppe 1] applied for asylum").

DECISION RULES Answer all four steps for the passage.
1. MARKER: Is there an explicit marker (a) to (d)? No if there is none, including facts and statistics only or implicit cues only.
2. ACTOR: Is [Gruppe 1] or a member the one who is warm toward others? No if a Step 2 item of NOT WARMTH applies.
3. NOT EXCLUDED: Is it true that no Step 3 item of NOT WARMTH applies?
4. CONFIDENCE: The label is TRUE if S1, S2 and S3 are all TRUE, otherwise FALSE. For a TRUE label, check the triggers below.

CONFIDENCE
A TRUE label needs an exact span from the passage that shows [Gruppe 1] or a member as the warm actor. If you cannot point to one, label FALSE. If a TRUE label has at least one trigger (T), keep TRUE and start the explanation with "NOT SURE:" and the trigger codes. With no trigger, do not hedge. T1 Dual reading: the key word may mean warmth or only competence, courage or utility ("brave", "reliable", "cooperative", "useful"). T2 Unclear intent: a real act is described, but its friendly, kind, honest, fair or altruistic nature is neither stated nor unmistakable (e.g. could be purely self-interested or routine duty). T3 Aid/duty wording: borderline whether an aid, rescue or public-service report conveys genuine prosocial warmth or only routine bureaucratic/professional execution. T4 Unverified/conditional only: the warm behavior is only hoped for, promised, or rumored ("expected to help", "allegedly donated") and neither stated as fact nor asserted by a quoted speaker. T5 Borderline explicit/implicit: it is borderline whether the evidence is an explicit marker or only implicit. Never use NOT SURE on FALSE. Unclear membership is FALSE, not NOT SURE.

OUTPUT FORMAT 
Return only a valid JSON object, with no text before or after it. The object has exactly two keys, in this order: "explanation": one string in this form: Summary | S1: T/F | S2: T/F | S3: T/F | IMPLICIT: T/F 

Summary: one short sentence naming the actor and the key words from the text in single quotes. For NOT SURE, start the whole string with "NOT SURE: T1" (all trigger codes that apply) and then the sentence. 

S1: a marker was found. S2: [Gruppe 1] or a member is the actor. S3: no Step 3 exclusion applies. IMPLICIT: T if the strongest evidence is implicit, F if it is explicit or there is none. 
Fill all four fields even when the label is FALSE. Use single quotes only inside the string. "label": "TRUE" if S1, S2 and S3 are all T, otherwise "FALSE".

EXAMPLES 
Passage: The [Gruppe 1] native collected donations for the refugees and welcomed them warmly. {"explanation": "The [Gruppe 1] native 'collected donations for the refugees and welcomed them warmly'. | S1: T | S2: T | S3: T | IMPLICIT: F", "label": "TRUE"}

Passage: Partners described the [Gruppe 1] negotiator as reliable and cooperative in talks. {"explanation": "NOT SURE: T1 Partners call the [Gruppe 1] negotiator 'reliable and cooperative', which may mean competence rather than warmth. | S1: T | S2: T | S3: T | IMPLICIT: F", "label": "TRUE"}

Passage: The local population showed great readiness to help the [Gruppe 1]. {"explanation": "The local population 'showed great readiness to help the [Gruppe 1]', so the warmth is toward [Gruppe 1], not from it. | S1: T | S2: F | S3: T | IMPLICIT: F", "label": "FALSE"}

Passage: The brave [Gruppe 1] workers strengthen the economy. {"explanation": "The [Gruppe 1] workers are described as 'brave' and 'strengthen the economy', reflecting competence and a benefit frame rather than warmth. | S1: F | S2: T | S3: F | IMPLICIT: F", "label": "FALSE"}

"""

instructions = ANNOTATION_PROMPT
reminder = "Ensure you strictly evaluate only the explicit or clearly implied warmth attributes of [Gruppe 1] within the text. Do not code warmth based on the actions of other groups"
json_mode = True

print("--- Instructions to annotate WARMTH Loaded ---")
json_mode= True


--- Instructions to annotate WARMTH Loaded ---


# STEP 3: Paragraph Input Data

In [9]:
if "API_KEY" not in globals():
    raise RuntimeError("Please run STEP 1 first.")

INPUT_EXCEL_FILE = "sample_5k_translated.xlsx"
CHECKPOINT_FILE = "warmth_e_annotation_checkpoint.jsonl"
OUTPUT_EXCEL = "sample_5k_warmth_e_annotated.xlsx"
OUTPUT_JSON = "sample_5k_warmth_e_annotated.json"

try:
    sample = pd.read_excel(INPUT_EXCEL_FILE)
    if "row_id" not in sample.columns:
        sample.insert(0, "row_id", range(len(sample)))

    # Preprocessing: Exclude the group 'JUDE' from input data
    initial_len = len(sample)
    group_col = None
    for col in ["group", "target_group", "target", "group_name", "Gruppe", "gruppe"]:
        if col in sample.columns:
            group_col = col
            break

    if group_col:
        sample = sample[sample[group_col].astype(str).str.strip().str.upper() != "JUDE"].reset_index(drop=True)
        print(f"Preprocessing: Excluded {initial_len - len(sample)} rows where '{group_col}' was 'JUDE'.")
    else:
        for col in sample.select_dtypes(include=["object"]).columns:
            sample = sample[sample[col].astype(str).str.strip().str.upper() != "JUDE"]
        sample = sample.reset_index(drop=True)
        print(f"Preprocessing: Excluded 'JUDE' across text columns ({initial_len - len(sample)} rows removed).")

    print(f"Successfully loaded {len(sample)} paragraph(s) from '{INPUT_EXCEL_FILE}'.")
except FileNotFoundError:
    raise FileNotFoundError(f"Error: Could not find '{INPUT_EXCEL_FILE}' in the current folder.")

Preprocessing: Excluded 189 rows where 'group' was 'JUDE'.
Successfully loaded 4811 paragraph(s) from 'sample_5k_translated.xlsx'.


# STEP 4: Set up LLM API Call Function

In [10]:
if "API_KEY" not in globals():
    raise RuntimeError("Please run STEP 1 first.")

def call_llm(text, instructions, reminder=None, max_retries=MAX_RETRIES):
    user_msg = f"Text: '{text}'"
    if reminder:
        user_msg += f"\nReminder: {reminder}"
        
    body = {
        "model": MODEL,
        "messages": [
            {"role": "system", "content": instructions},
            {"role": "user",   "content": user_msg},
        ],
        "temperature": TEMPERATURE,
        "seed": SEED,
        "reasoning_effort": "none",
        "response_format": {"type": "json_object"},
        "max_tokens": 1000,
    }
    
    last_error = None
    if json_mode:
        body["response_format"] = {"type": "json_object"}

        
    for attempt in range(1, max_retries + 1):
        try:
            r = requests.post(f"{HOST}/chat/completions",
                              headers={"Authorization": f"Bearer {API_KEY}"},
                              json=body, timeout=TIMEOUT)
                                             
            r.raise_for_status()
            j = r.json()
            return {
                "output": j["choices"][0]["message"]["content"].strip(),
                "tokens_prompt": j["usage"]["prompt_tokens"],
                "tokens_response": j["usage"]["completion_tokens"],
                "error": None,
            }
            
        except requests.exceptions.HTTPError as he:
            last_error = str(he)
            if he.response.status_code == 504:
                # 504 Gateway Timeout: back off aggressively 
                wait_time = 15 * (2 ** (attempt - 1))
                print(f"Attempt {attempt}/{max_retries} failed: 504 Gateway Timeout. Server is overloaded. Waiting {wait_time}s...")
                time.sleep(wait_time)
            else:
                print(f"Attempt {attempt}/{max_retries} failed with HTTP error: {he}")
                time.sleep(5 * attempt)
                # Back off softly if too many requests are sent

            if r.status_code == 429:
                                wait_time = 60
                                print(f"Attempt {attempt}/{max_retries} failed: 429 Too Many Requests. Cooling down for {wait_time}s...")
                                time.sleep(wait_time)
                                continue

            if he.response.status_code is not None and 400 <= he.response.status_code < 500 and he.response.status_code not in (408, 429):
                break

        except Exception as e:
            last_error = str(e)
            print(f"Attempt {attempt}/{max_retries} failed (Network error): {e}")
            time.sleep(5 * attempt)
            
    print("All retry attempts failed for this entry.")
    return {"output": None, "tokens_prompt": None, "tokens_response": None, "error": last_error}

# STEP 5: Run Annotation Loop with parallelization 

In [11]:
if "API_KEY" not in globals() or "sample" not in globals():
    raise RuntimeError("Please run STEP 1 and STEP 3 first.")

print(f"Starting annotation loop for {len(sample)} entries...")

# 1. Load existing checkpoints if any (enables resuming if interrupted)
completed_ids = set()
results_cache = {}

if os.path.exists(CHECKPOINT_FILE):
    with open(CHECKPOINT_FILE, "r", encoding="utf-8") as f:
        for line in f:
            try:
                data = json.loads(line)
                row_id = data.get("row_id")
                if row_id is not None:
                    completed_ids.add(row_id)
                    results_cache[row_id] = data
            except json.JSONDecodeError:
                continue
    print(f"Resumed from checkpoint: {len(completed_ids)} already annotated entries found.")

# Thread-safe lock for writing to the checkpoint file
file_lock = threading.Lock()

def process_row(row):
    row_id = row["row_id"]
    if row_id in completed_ids:
        return results_cache[row_id]
    
    # Explicitly grab text from the 'translation' column
    text_content = str(row.get("translation", ""))
    
    response = call_llm(text_content, instructions, reminder=reminder)
    
    # Parse the LLM output JSON string
    parsed_output = {"label": None, "explanation": "Failed to parse JSON"}
    
    if response["output"]:
        try:
            clean_out = response["output"].strip()
            if clean_out.startswith("```json"):
                clean_out = clean_out[7:]
            if clean_out.startswith("```"):
                clean_out = clean_out[3:]
            if clean_out.endswith("```"):
                clean_out = clean_out[:-3]
            
            parsed_json = json.loads(clean_out.strip())
            parsed_output["label"] = parsed_json.get("label")
            parsed_output["explanation"] = parsed_json.get("explanation")
        except Exception as e:
            parsed_output["explanation"] = f"JSON parse error: {e} | Raw: {response['output']}"

    result_entry = {
        "row_id": row_id,
        "label": parsed_output["label"],
        "explanation": parsed_output["explanation"],
        "raw_output": response["output"],
        "error": response["error"],
        "tokens_prompt": response["tokens_prompt"],
        "tokens_response": response["tokens_response"]
    }
    
    with file_lock:
        with open(CHECKPOINT_FILE, "a", encoding="utf-8") as cp:
            cp.write(json.dumps(result_entry, ensure_ascii=False) + "\n")
            
    return result_entry

# 2. Run execution using ThreadPoolExecutor
results = []
errors_in_a_row = 0

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(process_row, row): row for _, row in sample.iterrows()}
    
    completed_count = len(completed_ids)
    total_count = len(sample)
    
    for future in as_completed(futures):
        res = future.result()
        results.append(res)
        completed_count += 1
        
        if res.get("error"):
            errors_in_a_row += 1
        else:
            errors_in_a_row = 0
            
        if errors_in_a_row >= STOP_AFTER_ERRORS:
            print(f"Stopping early: {STOP_AFTER_ERRORS} errors encountered in a row.")
            break
            
        if completed_count % 10 == 0 or completed_count == total_count:
            print(f"Progress: {completed_count}/{total_count} processed.")

# 3. Merge results back into DataFrame and save final outputs
results_df = pd.DataFrame(results)
final_df = pd.merge(sample, results_df, on="row_id", how="left")

final_df.to_excel(OUTPUT_EXCEL, index=False)
final_df.to_json(OUTPUT_JSON, orient="records", indent=2, force_ascii=False)

print("\n--- Annotation Complete! ---")
print(f"Checkpoint saved to: {CHECKPOINT_FILE}")
print(f"Excel output saved to: {OUTPUT_EXCEL}")
print(f"JSON output saved to: {OUTPUT_JSON}")

Starting annotation loop for 4811 entries...
Progress: 10/4811 processed.
Progress: 20/4811 processed.
Progress: 30/4811 processed.
Progress: 40/4811 processed.
Progress: 50/4811 processed.
Progress: 60/4811 processed.
Progress: 70/4811 processed.
Progress: 80/4811 processed.
Progress: 90/4811 processed.
Progress: 100/4811 processed.
Progress: 110/4811 processed.
Progress: 120/4811 processed.
Progress: 130/4811 processed.
Progress: 140/4811 processed.
Progress: 150/4811 processed.
Progress: 160/4811 processed.
Progress: 170/4811 processed.
Progress: 180/4811 processed.
Progress: 190/4811 processed.
Progress: 200/4811 processed.
Progress: 210/4811 processed.
Progress: 220/4811 processed.
Progress: 230/4811 processed.
Progress: 240/4811 processed.
Progress: 250/4811 processed.
Progress: 260/4811 processed.
Progress: 270/4811 processed.
Progress: 280/4811 processed.
Progress: 290/4811 processed.
Progress: 300/4811 processed.
Progress: 310/4811 processed.
Progress: 320/4811 processed.
Prog